# RNN for review

Як вже зазначалось вище, модель RNN розрізняє послідовність слів, в цьому випадку, вона навіть дуже важлива, щоб модель спіймала контекст.  
Тому стоп-слова лишимо як є.  
І для векторизації буде використана TextVectorization.  
Цей інстрмент було створено спеціально для нейромереж.  
На виході ми отримаємо тензор з числами, які будуть відповідати ID слів у відгуку.  
Також таким чином буде збережено порядок цих слів, що є ключовим для RNN.

Проаналізуємо розмір відгуків.

In [7]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Input, GRU, TextVectorization, Embedding, Bidirectional, Dropout

import joblib

In [8]:
train_ds = pd.read_csv('train_test_val/train/train_rnn.csv')

In [9]:
train_ds.head()

,Rating,Recommended,full_review,clean_text
0,5,1,Excellent I bought this in store on sale and c...,excellent I buy this in store on sale and comp...
1,5,1,Magnificent shirt!!! I have several of goodhyo...,magnificent shirt I have several of goodhyouma...
2,5,1,Cute and versatile Picked this up at the local...,cute and versatile pick this up at the local r...
3,2,0,"Not if you're busty I am a 32dd, 5'4'' and 125...",not if you re busty I be a dd and lbs and th...
4,4,1,Neutral blue I tried this on from the sale sec...,neutral blue I try this on from the sale secti...


In [10]:
train_ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10870 entries, 0 to 10869
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Rating       10870 non-null  int64 
 1   Recommended  10870 non-null  int64 
 2   full_review  10870 non-null  object
 3   clean_text   10870 non-null  object
dtypes: int64(2), object(2)
memory usage: 339.8+ KB


In [11]:
text_length = [len(text.split()) for text in train_ds['clean_text']]
mean_length = np.mean(text_length)
max_length = np.max(text_length)
min_length = np.min(text_length)
percent = np.percentile(text_length, 95)

print(f'Mean: {mean_length:.2f}'
      f'\nMax: {max_length}'
      f'\nMin: {min_length}'
      f'\n95%: {percent}'
)

Mean: 62.18
Max: 121
Min: 2
95%: 105.0


Бачимо, що 95% наших відгуків це доволі обширні тексти, що дивно, стільки писати у відгуку.  
Максимальна довжина тексту 121 слово, а 95% це 105 слів.  
не бачу сенсу якось обрізати для векторизації, тому лишу максимальну кількість слів.

In [25]:
max_tokens = 20000
sequence_length = max_length.item()

In [26]:
text_vectorization = TextVectorization(
    # standardize=None,
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

In [27]:
text_vectorization.adapt(train_ds['clean_text'])

In [28]:
X_train = text_vectorization(train_ds['clean_text'])
X_train.shape

TensorShape([10870, 121])

In [29]:
X_train_np = np.array(X_train)

In [30]:
idx = np.arange(len(X_train_np))
idx_tr, idx_val = train_test_split(idx, test_size=0.2, random_state=16)

In [31]:
y_rate = train_ds['Rating']
y_rate.shape

(10870,)

In [32]:
y_rec = train_ds['Recommended']
y_rec.shape

(10870,)

In [33]:
y_rec_reset = y_rec.reset_index(drop=True)
y_rate_reset = y_rate.reset_index(drop=True)

In [34]:
y_rate_np = y_rate_reset.to_numpy().reshape(-1, 1).astype('float32')
y_rec_np  = y_rec_reset.to_numpy().reshape(-1, 1).astype('float32')

In [35]:
shared_input = Input(shape=(sequence_length,), name='input')
embedding = Embedding(input_dim=max_tokens, output_dim=128, name='embedding')(shared_input)
lstm_out = Bidirectional(LSTM(64, name='LSTM'))(embedding)
shared_dense = Dense(64, activation='relu', name='shared_dense')(lstm_out)
shared_dense = Dropout(0.3)(shared_dense)

rate_out = Dense(1, activation='linear', name='rating')(shared_dense)

rec_out = Dense(1, activation='sigmoid', name='recommendation')(shared_dense)


model_lstm = tf.keras.Model(inputs=shared_input, outputs=[rate_out, rec_out])

model_lstm.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)            │ (None, 121)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 121, 128)          │       2,560,000 │ input[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional_1               │ (None, 128)               │          98,816 │ embedding[0][0]            │
│ (Bidirectional)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ shared_dense (Dense)          │ (None, 64)                │           8,256 │ bidirectional_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 64)                │               0 │ shared_dense[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rating (Dense)                │ (None, 1)                 │              65 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ recommendation (Dense)        │ (None, 1)                 │              65 │ dropout_1[0][0]            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,667,202 (10.17 MB)

 Trainable params: 2,667,202 (10.17 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss={
        'rating' : 'mse',
        'recommendation': 'binary_crossentropy'
    },
    metrics={
        'rating' : 'mae',
        'recommendation': 'accuracy'
    }
)

In [37]:
history_lstm = model_lstm.fit(X_train_np[idx_tr],
                              {'rating': y_rate_np[idx_tr], 'recommendation': y_rec_np[idx_tr]},
                              epochs=6,
                              batch_size=64,
                              validation_data=(
                              X_train_np[idx_val],
                              {'rating': y_rate_np[idx_val], 'recommendation': y_rec_np[idx_val]}
                              )
                            )

Epoch 1/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 14s 83ms/step - loss: 2.5595 - rating_loss: 2.0841 - rating_mae: 1.0820 - recommendation_accuracy: 0.7888 - recommendation_loss: 0.4744 - val_loss: 0.9901 - val_rating_loss: 0.6683 - val_rating_mae: 0.6025 - val_recommendation_accuracy: 0.8330 - val_recommendation_loss: 0.3217
Epoch 2/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - loss: 1.0843 - rating_loss: 0.8241 - rating_mae: 0.7131 - recommendation_accuracy: 0.8913 - recommendation_loss: 0.2601 - val_loss: 0.8240 - val_rating_loss: 0.5901 - val_rating_mae: 0.6056 - val_recommendation_accuracy: 0.9020 - val_recommendation_loss: 0.2337
Epoch 3/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - loss: 0.9185 - rating_loss: 0.7115 - rating_mae: 0.6655 - recommendation_accuracy: 0.9193 - recommendation_loss: 0.2068 - val_loss: 0.7559 - val_rating_loss: 0.5290 - val_rating_mae: 0.5636 - val_recommendation_accuracy: 0.9034 - val_recommendation_loss: 0.2269
Epoch 4/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/s

In [38]:
loss, rating_loss, recommendation_loss, rating_mae, recommendation_accuracy = model_lstm.evaluate(X_train_np[idx_val], {'rating': y_rate_np[idx_val], 'recommendation': y_rec_np[idx_val]})
print(f'Loss: {loss:.3f}'
      f'\nRating Loss: {rating_loss:.3f}'
      f'\nRating MAE: {rating_mae:.3f}'
      f'\nRecomendation Accuracy: {recommendation_accuracy:.3f}'
      f'\nRecomendation Loss: {recommendation_loss:.3f}'
      )

68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.8388 - rating_loss: 0.5680 - rating_mae: 0.5919 - recommendation_accuracy: 0.8928 - recommendation_loss: 0.2703
Loss: 0.839
Rating Loss: 0.568
Rating MAE: 0.592
Recomendation Accuracy: 0.893
Recomendation Loss: 0.270


In [39]:
gru_input = Input(shape=(sequence_length,), name='gru_input')
gru_embedding = Embedding(input_dim=max_tokens, output_dim=128, name='embedding')(gru_input)

gru_out = GRU(64, dropout=0.2, name='gru')(gru_embedding)
                        
gru_dense = Dense(64, activation='relu', name='gru_dense')(gru_out)
gru_dense = Dropout(0.3)(gru_dense)

gru_rate_out = Dense(1, activation='linear', name='rating')(gru_dense)

gru_rec_out = Dense(1, activation='sigmoid', name='recommendation')(gru_dense)

model_gru = tf.keras.Model(inputs=gru_input, outputs=[gru_rate_out, gru_rec_out])

model_gru.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gru_input (InputLayer)        │ (None, 121)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 121, 128)          │       2,560,000 │ gru_input[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gru (GRU)                     │ (None, 64)                │          37,248 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gru_dense (Dense)             │ (None, 64)                │           4,160 │ gru[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 64)                │               0 │ gru_dense[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rating (Dense)                │ (None, 1)                 │              65 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ recommendation (Dense)        │ (None, 1)                 │              65 │ dropout_2[0][0]            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,601,538 (9.92 MB)

 Trainable params: 2,601,538 (9.92 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model_gru.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss={
        'rating' : 'mse',
        'recommendation': 'binary_crossentropy'
    },
    metrics={
        'rating' : 'mae',
        'recommendation': 'accuracy'
    }
)

In [41]:
history_gru = model_gru.fit(X_train_np[idx_tr],
                              {'rating': y_rate_np[idx_tr], 'recommendation': y_rec_np[idx_tr]},
                              epochs=6,
                              batch_size=64,
                              validation_data=(
                              X_train_np[idx_val],
                              {'rating': y_rate_np[idx_val], 'recommendation': y_rec_np[idx_val]}
                              )
                            )

Epoch 1/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 10s 61ms/step - loss: 3.1083 - rating_loss: 2.5963 - rating_mae: 1.2375 - recommendation_accuracy: 0.8026 - recommendation_loss: 0.5113 - val_loss: 1.8396 - val_rating_loss: 1.3566 - val_rating_mae: 0.9930 - val_recommendation_accuracy: 0.8123 - val_recommendation_loss: 0.4833
Epoch 2/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - loss: 2.0084 - rating_loss: 1.5235 - rating_mae: 0.9867 - recommendation_accuracy: 0.8223 - recommendation_loss: 0.4849 - val_loss: 1.6433 - val_rating_loss: 1.1681 - val_rating_mae: 0.8803 - val_recommendation_accuracy: 0.8123 - val_recommendation_loss: 0.4755
Epoch 3/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - loss: 1.5034 - rating_loss: 1.1018 - rating_mae: 0.8200 - recommendation_accuracy: 0.8323 - recommendation_loss: 0.4012 - val_loss: 0.8438 - val_rating_loss: 0.5649 - val_rating_mae: 0.5534 - val_recommendation_accuracy: 0.8666 - val_recommendation_loss: 0.2790
Epoch 4/6
136/136 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/ste

In [42]:
loss_gru, rating_loss_gru, recommendation_loss_gru, rating_mae_gru, recommendation_accuracy_gru = model_gru.evaluate(X_train_np[idx_val], {'rating': y_rate_np[idx_val], 'recommendation': y_rec_np[idx_val]})
print(f'Loss: {loss_gru:.3f}'
      f'\nRating Loss: {rating_loss_gru:.3f}'
      f'\nRating MAE: {rating_mae_gru:.3f}'
      f'\nRecomendation Accuracy: {recommendation_accuracy_gru:.3f}'
      f'\nRecomendation Loss: {recommendation_loss_gru:.3f}'
      )

68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.6901 - rating_loss: 0.4737 - rating_mae: 0.4751 - recommendation_accuracy: 0.9089 - recommendation_loss: 0.2162
Loss: 0.690
Rating Loss: 0.474
Rating MAE: 0.475
Recomendation Accuracy: 0.909
Recomendation Loss: 0.216


In [24]:
# joblib.dump(model_lstm, 'model_lstm.joblib')
# joblib.dump(model_gru, 'model_gru.joblib')

['model_gru.joblib']

In [25]:
# joblib.dump(text_vectorization, 'vectorization_rnn.joblib')

['vectorization_rnn.joblib']

# Submission

* `model_gru` - для класифікації рекомендації та для визначення рейтинг за відгуком  
* `model_nr_rec` - для класифікації рекомендації БЕЗ відгука  
* `model_nr_rat` - для визначення рейтингу БЕЗ відгука  

In [26]:
model_nr_rat = joblib.load('models/no_review/model_nr_rat.joblib')
model_nr_rec = joblib.load('models/no_review/model_nr_rec.joblib')

In [3]:
rnn_test = pd.read_csv('train_test_val/test/test_rnn_review.csv')
rec_nr_rnn = pd.read_csv('train_test_val/test/test_rec_enc.csv')
rat_nr_rnn = pd.read_csv('train_test_val/test/test_rat_enc.csv')

In [6]:
rnn_test['clean_text'].shape

(9395,)

In [28]:
rnn_test.head()

,Id,clean_text
0,21403,magnificent clothe in contrast to the other re...
1,22553,shapeless tent I try this on in the store and ...
2,17436,versatile and then some I think this be a fun ...
3,4293,so simple but so cute I buy the multicolor str...
4,20149,magnificent simple tank the wide strap style b...


In [29]:
rec_nr_rnn.head()

,Age,Pos_Feedback_Cnt,Division,Department,Product_Category
0,0.813273,0.277647,0.813338,0.806434,0.806434
1,0.649031,-0.084574,0.813338,0.806434,0.806434
2,1.305999,-0.265685,0.813338,0.852045,0.831122
3,0.402667,-0.265685,0.813338,0.806434,0.806434
4,0.238425,-0.446795,0.853982,0.855630,0.888740


In [30]:
rat_nr_rnn.head()

,Age,Pos_Feedback_Cnt,Division,Department,Product_Category
0,0.813273,0.277647,4.176844,4.145440,4.145440
1,0.649031,-0.084574,4.176844,4.145440,4.145440
2,1.305999,-0.265685,4.176844,4.318103,4.294331
3,0.402667,-0.265685,4.176844,4.145440,4.145440
4,0.238425,-0.446795,4.308628,4.308951,4.369983


In [31]:
isna_mask = rnn_test['clean_text'].isna()

In [32]:
X_rev = text_vectorization(rnn_test.loc[~isna_mask, 'clean_text'])

In [33]:
y_rec = pd.Series(index=rnn_test.index, dtype=float)
y_rat = pd.Series(index=rnn_test.index, dtype=float)

In [34]:
X_nr_rec = rec_nr_rnn.loc[isna_mask]
X_nr_rat = rat_nr_rnn.loc[isna_mask]

In [35]:
rnn_pred = model_gru.predict(X_rev)

283/283 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step


In [36]:
rnn_pred_rat = rnn_pred[0].flatten()
rnn_pred_rec = rnn_pred[1].flatten()

In [37]:
y_rat[~isna_mask] = rnn_pred_rat
y_rec[~isna_mask] = rnn_pred_rec

In [38]:
y_rec[isna_mask] = model_nr_rec.predict(X_nr_rec)
y_rat[isna_mask] = model_nr_rat.predict(X_nr_rat)

In [39]:
y_rec = [ 1 if i > 0.5 
          else 0 for i in y_rec]

In [40]:
y_rat = [5 if i >= 4.5 
         else 4 if 3.5 <= i < 4.5 
         else 3 if 2.5 <= i < 3.5 
         else 2 if 1.5 <= i < 2.5 
         else 1 
         for i in y_rat]

In [41]:
sub_ds = pd.read_csv('final-project-danit-ds-3-4/sample_submission.csv')
sub_ds.head()

,Id,Rating,Recommended
0,21403,2,1
1,22553,5,0
2,17436,1,1
3,4293,1,0
4,20149,4,0


In [42]:
sub_ds['Rating'] = y_rat
sub_ds['Recommended'] = y_rec

In [43]:
sub_ds.head()

,Id,Rating,Recommended
0,21403,4,1
1,22553,2,0
2,17436,5,1
3,4293,5,1
4,20149,4,1


In [ ]:
sub_ds.to_csv('submission.csv', index=False)